# Filamentation SiO2 — 1030 nm, 10 µJ, waist en z = 0

Paramètres expérimentaux du 16 h 22 : sonde **515 nm**, **10 µJ**, profil 1/e²
`sx/sy = 11.5/11.0 µm` (diamètre équivalent 22.5 µm). Solveur
`sim/filament_sim.py` du dépôt, Éq. (2) de Couairon.

> **Le profil mesuré tranche la question du waist, et corrige ce que j'avais
> déduit.** Le rayon équivalent est `√(11.5×11.0) = 11.25 µm`, pas les 2.84 µm
> que j'avais obtenus en supposant un grandissement ×9 sur la caméra du
> profilomètre. Cette hypothèse était fausse. Recalé sur la mesure directe, mon
> ajustement de caustique (`w0 = 8.44 px`) donne `p = 1.333 µm/px`, et alors
> `M² = 0.81` avec `z` en µm — soit 1.00 à 19 % près, cohérent avec un
> faisceau quasi limité par diffraction compte tenu des ±9 % sur `zR`. Tout se
> referme.
>
> **Conséquence directe : ta demande initiale redevient valide.** À `w0 = 11.25 µm`
> et 10 µJ, l'intensité au waist vaut `1.74×10¹³ W/cm²`, soit **2.9× sous le
> clampage**. Le faisceau peut donc être lancé exactement à son waist, sans
> ioniser le milieu à l'entrée. C'était impossible à `w0 = 3 µm`.
>
> Il reste un ajustement : `z_R = 559 µm` et `L_c ≈ 83–99 µm`, donc **60 µm ne
> suffisent pas** — la boîte va à 300 µm pour contenir le collapse et les
> cycles qui suivent.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.constants import c as c_SI, epsilon_0, m_e, elementary_charge as q_e

for p in (Path.cwd().parent / "sim", Path.cwd() / "sim"):
    sys.path.insert(0, str(p))

from filament_sim import run, FIELD_TOGGLES, n_sellmeier
import figures_filament as ff

OUT_ROOT = Path("runs_z0"); OUT_ROOT.mkdir(exist_ok=True)
FIG_DIR = OUT_ROOT / "figures"; FIG_DIR.mkdir(exist_ok=True)
print("toggles disponibles :", FIELD_TOGGLES)

## 1. Paramètres

> **Le profil 11.5/11.0 µm n'est pas le waist dans l'échantillon.** Tu l'as vu
> toi-même sur l'axe z : la simulation étalait l'ionisation sur 300 µm là où
> tes images montrent un canal de quelques dizaines de µm. La raison est
> directe — à `w0 = 11.25 µm`, `z_R = 560 µm`, donc le faisceau **ne focalise
> pas** sur la boîte : l'intensité reste quasi constante et le milieu s'ionise
> partout au lieu de le faire localement.
>
> Et ça donne une mesure de `w0` gratuite, indépendante de toute calibration
> caméra : **la longueur du canal *est* z_R**.
>
> | canal observé | `w0` impliqué |
> |---|---|
> | 40 µm | 3.0 µm |
> | 60 µm | 3.7 µm |
> | 80 µm | 4.3 µm |
>
> Ce qui recoupe mon ajustement de caustique (2.84 µm) et exclut 11.25 µm.
> Ajuste `CHANNEL_LENGTH_UM` sur ce que tu lis vraiment sur tes images.

**Conséquence sur la boîte.** À `w0 ≈ 3 µm`, l'intensité au waist vaut
`2.4×10¹⁴ W/cm² = 4.9× le clampage` : on ne peut pas non plus démarrer au
waist. Il faut `|begin| > 78 µm` pour passer sous `I_clamp`, ~193 µm pour être
tranquille. D'où `begin = −200 µm` (entrée à 15.4 µm de rayon,
`9.3×10¹² W/cm²`, 5.4× sous le seuil).

Marburger prédit alors le foyer non-linéaire vers **z ≈ −104 µm**, et un canal
de l'ordre de `z_R = 40 µm` autour. La fenêtre de tracé (`Z_LIM`) se cadre
dessus pour que les axes correspondent aux tiennes.

In [ ]:
# ---- laser (mesures du 16h22) ----
WAVELENGTH_M = 1030e-9
SX_UM, SY_UM = 11.5, 11.0                      # rayons 1/e^2 du profil fourni
# CE PROFIL N'EST PAS LE WAIST DANS L'ECHANTILLON -- voir markdown ci-dessus.
# w0 est fixe par la LONGUEUR DU CANAL observe, qui vaut z_R = pi w0^2 n0/lambda.
CHANNEL_LENGTH_UM = 40.0                       # <-- lu sur tes images
W0_M = ff.w0_from_channel_length(CHANNEL_LENGTH_UM, 1030e-9, 1.45)*1e-6
DELTA_T_S    = 263e-15                         # A CONFIRMER (non redonnee)
ENERGY_INCIDENT_UJ = 10.0
ENERGY_IN_GLASS_UJ = ENERGY_INCIDENT_UJ * (1 - ((1.45-1)/(1.45+1))**2)   # 9.663 uJ

# ---- materiau ----
N2          = 2.74e-20        # Milam 1998 @1053nm
UI_EV       = 9.0
MEFF_REL    = 0.64
TAU_C_S     = 1.7e-15
TAU_R_S     = 330e-15         # piegeage STE (Mouskeftaras 2013)
TAU_STE_S   = 1e-12           # decroissance STE (Sakurai) ; None = STE geles
RHO_MAX_CM3 = 2.1e22
US_EV       = 6.0
E_TR_EV     = 4.2             # resonance Lorentz STE (Mao et al.)
F_R, TAU_D_S, TAU_S_S = 0.18, 32e-15, 12e-15

# ---- sonde ----
LAMBDA_PROBE_M = 515e-9       # etait 490 nm

# ---- boite : z=0 = waist geometrique, on demarre EN AMONT ----
# A w0~3 um l'intensite au waist vaut 4.9x le clampage : impossible d'y
# demarrer. Il faut |begin| > 78 um pour passer sous I_clamp, ~193 um pour
# etre confortable. check_entrance_intensity le revalide a chaque lancement.
BEGIN_M, END_M = -200e-6, 60e-6

# ---- deux presets ----
FAST = dict(Nz=7000,  Nt=2048, Nr=1024, R_factor=25.0,
            save_stride=20, rho_t_stride=16, rho_r_stride=2)
PROD = dict(Nz=14000, Nt=4096, Nr=2048, R_factor=25.0,
            save_stride=40, rho_t_stride=16, rho_r_stride=4)
GRID = FAST                      # <-- bascule ici

n0 = n_sellmeier(WAVELENGTH_M)
k0 = 2*np.pi*n0/WAVELENGTH_M
zR = k0*W0_M**2/2
tp = DELTA_T_S/np.sqrt(2*np.log(2)); tmax = 5*tp
dt = 2*tmax/GRID["Nt"]; dz = (END_M-BEGIN_M)/GRID["Nz"]
R = GRID["R_factor"]*W0_M; dr = R/(GRID["Nr"]-1)
n_saves = GRID["Nz"]//GRID["save_stride"]+1
Nt_sub = (GRID["Nt"]-1)//GRID["rho_t_stride"]+1
Nr_sub = (GRID["Nr"]-2)//GRID["rho_r_stride"]+1

print(f"w0 = {W0_M*1e6:.2f} um (equivalent de {SX_UM}/{SY_UM})   z_R = {zR*1e6:.0f} um")
print(f"energie : {ENERGY_INCIDENT_UJ} uJ -> {ENERGY_IN_GLASS_UJ:.3f} uJ dans le verre")
print(f"z : Nz={GRID['Nz']}, dz={dz*1e9:.1f} nm | {n_saves} plans, dz_save={dz*GRID['save_stride']*1e6:.2f} um")
print(f"r : R_max={R*1e6:.0f} um, dr={dr*1e9:.0f} nm, {W0_M/dr:.0f} pts dans w0, {1e-6/dr:.0f} dans 1 um")
print(f"t : Nt={GRID['Nt']}, dt={dt*1e15:.2f} fs, fenetre +/-{tmax*1e15:.0f} fs, f_Nyq/f0={1/(2*dt)/(c_SI/WAVELENGTH_M):.2f}")
print(f"cube : {3*n_saves*Nr_sub*Nt_sub*4/1e6:.0f} Mo")

## 2. Où le collapse est-il attendu ?

Le faisceau démarre à son waist, donc `L_DF = k w0²/2 = z_R` directement —
pas de correction de rayon d'entrée cette fois, et pas de focalisation
externe (`f_ext = None`).

In [ ]:
print("=== CONTROLE : intensite au plan d'entree ===")
I_in, w_in, z_safe = ff.check_entrance_intensity(
    ENERGY_IN_GLASS_UJ, W0_M, DELTA_T_S, BEGIN_M, WAVELENGTH_M, n0)

print("\n=== P_cr et longueur de collapse ===")
P_cr = ff.critical_power(N2, WAVELENGTH_M, n0)
P_in = ENERGY_IN_GLASS_UJ*1e-6/(tp*np.sqrt(np.pi/2))
# L_DF doit utiliser le rayon AU PLAN D'ENTREE, pas le waist focal, et la
# focalisation externe est la distance entree->waist.
ratio, L_DF, L_c, L_cf = ff.marburger_collapse(P_in, P_cr, w_in, WAVELENGTH_M,
                                                n0, f_ext=abs(BEGIN_M))
Z_NL_UM = L_cf*1e6 + BEGIN_M*1e6
print(f"P_cr={P_cr*1e-6:.2f} MW  P_in/P_cr={ratio:.1f}")
print(f"L_DF={L_DF*1e6:.0f} um  L_c={L_c*1e6:.0f} um  L_c,f={L_cf*1e6:.0f} um")
print(f"-> foyer non-lineaire attendu vers z = {Z_NL_UM:+.0f} um "
      f"(waist geometrique en z=0)")

# Fenetre de trace : cadree sur le foyer non-lineaire, largeur ~2 z_R,
# pour que l'axe long corresponde a celui de tes figures experimentales.
Z_R_UM = zR*1e6
Z_LIM = (Z_NL_UM - Z_R_UM, Z_NL_UM + Z_R_UM)
print(f"z_R = {Z_R_UM:.0f} um -> fenetre de trace Z_LIM = "
      f"({Z_LIM[0]:.0f}, {Z_LIM[1]:.0f}) um")

## 3. Lancer

`FAST` vise l'itération rapide. Le coût est dominé par la transformée de
Hankel (matrice dense `Nr × Nr` appliquée 4× par pas), donc il croît en
`Nr²·Nt·Nz` — c'est `Nr` qu'il faut baisser en premier si c'est trop lent, pas
`Nz`.

In [ ]:
OUT_DIR = str(OUT_ROOT / f"z0_60um_{GRID['Nz']}x{GRID['Nr']}x{GRID['Nt']}")

res = ff.load_scenario_npz(OUT_DIR)
if res is None:
    print(f"Lancement -> {OUT_DIR}")
    res = run(
        Nz=GRID["Nz"], Nt=GRID["Nt"], Nr=GRID["Nr"], R_factor=GRID["R_factor"],
        begin=BEGIN_M, end=END_M,                 # waist en z=0
        save_stride=GRID["save_stride"], ckpt_every=200, verbose=True,
        wavelength=WAVELENGTH_M, energy_uJ=ENERGY_IN_GLASS_UJ,
        w0=W0_M, delta_t=DELTA_T_S,
        n2=N2, Ui_eV=UI_EV, meff_rel=MEFF_REL,
        tau_c=TAU_C_S, tau_r=TAU_R_S, rho_max=RHO_MAX_CM3,
        Us_eV=US_EV, tau_ste=TAU_STE_S,
        f_R=F_R, tau_d=TAU_D_S, tau_s=TAU_S_S,
        enable_ste=True, lambda_probe=LAMBDA_PROBE_M,
        rho_t_stride=GRID["rho_t_stride"], rho_r_stride=GRID["rho_r_stride"],
        out_dir=OUT_DIR, envelope="gaussian_focused",
    )
print(f"\nU_beam(0) attendu ~ {ENERGY_IN_GLASS_UJ:.2f} uJ")
ff.run_health_check(res, out_dir=OUT_DIR, label="z0_60um", rho_max=RHO_MAX_CM3)

## 4. Diagnostics standard

In [ ]:
VL = [(Z_NL_UM, "foyer non-lineaire (Marburger)", "tab:blue"),
      (0.0, "waist geometrique", "purple")]
ff.plot_fig7_fluence_contours(res, levels=(1.,5.,20.,50.), label="z0, 60 µm",
                              vlines=VL, save=str(FIG_DIR/"fluence.png"))
fig = ff.plot_fig8_peak_intensity({"z0, 60 µm": res})
for zv,l,cc in VL: fig.axes[0].axvline(zv, ls="--", color=cc, lw=1.2, label=l)
fig.axes[0].legend(fontsize=8); fig.savefig(FIG_DIR/"peak_intensity.png", dpi=150)

NC = epsilon_0*m_e*(2*np.pi*c_SI/LAMBDA_PROBE_M)**2/q_e**2*1e-6
ff.plot_free_vs_trapped_vs_z(res, rho_max_cm3=RHO_MAX_CM3, nc_probe_cm3=NC,
                             vlines=VL, save=str(FIG_DIR/"rho.png"))
ff.count_refocusing_cycles(res)

## 5. La planche pompe-sonde — à comparer à l'expérience

Même format que tes données : une colonne par délai, vue de face en haut,
vue de côté en bas, échelle de couleur commune.

**Comment les délais > 1.1 ps sont obtenus.** La fenêtre temporelle du
solveur vaut ±5 t_p = ±1117 fs. Au-delà, `probe_phase_map` n'extrapole pas
naïvement : à t = 5 t_p le champ vaut exp(−25) ≈ 10⁻¹¹ de son maximum, donc
les équations de population se réduisent à deux ODE linéaires
(ρ_e décroît en τ_r, alimente ρ_s qui décroît en τ_STE) dont la solution est
**analytique et exacte**. Vérifié contre une intégration numérique : accord à
10⁻⁸ %.

**Ce que ce modèle ne contient pas.** Aucune physique thermique ni acoustique.
Sur tes données à 3–9 ns on voit clairement une onde de choc qui s'étend :
elle ne sortira jamais de cette simulation. Jusqu'à ~2 ps en revanche, le
déphasage est porté par ρ_e, ρ_s et le Kerr, qui sont tous les trois dans le
modèle.

In [ ]:
# balayage fin de l'experience : -1 -> 2 ps. Les delais grossiers
# (3 ps -> 6 ns) sortent du modele : ni thermique ni acoustique dedans.
DELAYS_FS = [-500, 0, 250, 500, 1000, 1500, 2000]

ff.plot_delay_series(
    res, DELAYS_FS,
    lambda_probe_m=LAMBDA_PROBE_M, E_tr_eV=4.2, n2=N2,
    tau_r_s=TAU_R_S, tau_ste_s=TAU_STE_S,
    z_face_um=None,          # None = plan le plus intense (auto)
    x_half_um=15.0, z_lim=Z_LIM,
    save=str(FIG_DIR/"delay_series.png"),
);

## 5bis. Format expérimental : OPL (nm) et transmittance

Ta figure trace deux grandeurs que je ne produisais pas :

**L'OPL en nanomètres**, pas la phase en radians. C'est l'intégrale de Δn sur
la ligne de visée : `φ = 2π·OPL/λ`, soit **1 rad = 82 nm** à 515 nm. Ton
échelle ±15 nm correspond donc à **±0.183 rad** — cohérent avec les ~0.2 rad
que tu annonçais.

**La transmittance**, c'est-à-dire la partie *imaginaire* de l'indice, que je
ne calculais pas du tout. Elle vient de l'absorption par porteurs libres
(Bremsstrahlung inverse) évaluée **à la longueur d'onde sonde** :

`T = exp(−σ_sonde · ∫ρ_e dl)`,  σ = 3.11×10⁻¹⁸ cm² à 515 nm avec τ_c = 1.7 fs

Ordres de grandeur, pour situer :

| ρ_e (cm⁻³) | L = 2 µm | 6 µm | 20 µm |
|---|---|---|---|
| 10¹⁹ | 0.994 | 0.981 | 0.940 |
| **10²⁰** | 0.940 | **0.830** | 0.537 |
| 3×10²⁰ | 0.830 | 0.571 | 0.154 |

Ton échelle expérimentale 0.75–1.15 est donc compatible avec ρ_e ~ 10²⁰ sur
quelques µm — exactement la densité de clampage attendue. **C'est une
contrainte indépendante du déphasage**, et elle vaut mieux que lui pour caler
le modèle, parce qu'elle ne dépend ni du canal STE ni du Kerr.

Les STE ne contribuent pas à l'absorption ici : leur bande est à 5.2 eV et la
sonde à 515 nm ne fait que 2.41 eV, donc l'oscillateur de Lorentz y est
purement dispersif.

In [ ]:
# Format experimental : OPL [nm] + transmittance, vues dessus et cote.
PROBE_KW = dict(lambda_probe_m=LAMBDA_PROBE_M, E_tr_eV=E_TR_EV, n2=N2,
                tau_c_s=TAU_C_S, tau_r_s=TAU_R_S, tau_ste_s=TAU_STE_S)

fig, d = ff.plot_opl_panel(
    res, delay_fs=0.0,
    z_shift_um=-BEGIN_M*1e6,     # z=0 -> face d'entree, comme "z from interface"
    z_lim=None,                  # ou (0, 330) pour cadrer comme l'experience
    opl_clip_nm=15.0, t_lim=(0.75, 1.15), x_half_um=70.0,
    save=str(FIG_DIR/"opl_panel_0ps.png"), **PROBE_KW)
print(f"sigma_sonde = {d['sigma_cm2']:.3e} cm2   f_STE = {d['f_ste']:.4f}")
print(f"OPL max = {np.abs(d['opl_nm']).max():.2f} nm   "
      f"(experience : ~15 nm)")
print(f"transmittance min = {d['transmittance'].min():.3f}   "
      f"(experience : ~0.75)")

In [ ]:
# Serie de delais au format experimental
for dly in DELAYS_FS:
    fig, d = ff.plot_opl_panel(
        res, delay_fs=float(dly), z_shift_um=-BEGIN_M*1e6,
        opl_clip_nm=15.0, t_lim=(0.75, 1.15), x_half_um=70.0,
        title=f"simulation, 10 uJ, delay {dly/1000:+.3f} ps",
        save=str(FIG_DIR/f"opl_panel_{dly:+05.0f}fs.png"), **PROBE_KW)
    print(f"  {dly:+6.0f} fs : OPL max = {np.abs(d['opl_nm']).max():8.2f} nm, "
          f"T min = {d['transmittance'].min():.3f}")

### Décomposition par canal

Si l'amplitude ne colle pas à l'expérience (~0.2 rad au maximum), c'est ici
qu'on voit lequel des trois canaux en est responsable, plutôt que de deviner.

In [ ]:
for chans in (("drude",), ("ste",), ("kerr",), ("drude","ste","kerr")):
    _, _, phi = ff.probe_phase_map(res, 0.0, lambda_probe_m=LAMBDA_PROBE_M,
                                   E_tr_eV=E_TR_EV, n2=N2, tau_r_s=TAU_R_S,
                                   tau_ste_s=TAU_STE_S, include=chans, x_half_um=15.0)
    print(f"  a 0 fs, canaux {str(chans):32s} |phi|max = {np.abs(phi).max():6.3f} rad")
print()
for d in DELAYS_FS:
    _, _, phi = ff.probe_phase_map(res, d, lambda_probe_m=LAMBDA_PROBE_M, E_tr_eV=E_TR_EV,
                                    n2=N2, tau_r_s=TAU_R_S, tau_ste_s=TAU_STE_S, x_half_um=15.0)
    print(f"  delai {d:5d} fs : |phi|max = {np.abs(phi).max():6.3f} rad")
print("\nMesure experimentale : ~0.2 rad")

## 6. Itérer

Les trois leviers, par ordre d'effet attendu sur le déphasage :

| levier | comment |
|---|---|
| énergie | `ENERGY_IN_GLASS_UJ` |
| forme du faisceau | `W0_M` — la caustique mesurée donne un profil ~1.6–2.4× moins concentré qu'une gaussienne, donc un `w0` effectif jusqu'à 1.55× plus grand reproduit mieux l'intensité crête réelle |
| canal STE | `TAU_STE_S` (None = STE gelés) et `E_tr_eV` (4.2 dans le dépôt, 5.8 dans le slider — facteur 2.4 sur la contribution STE) |

Et pour isoler un terme physique, les six interrupteurs de l'Éq. (3) sont
disponibles : `run(..., enable_plasma_defocusing=False)` etc.

In [ ]:
# Exemple : meme run avec w0 effectif (profil non gaussien) -- decommenter
# W0_EFF = W0_M*1.55
# OUT_B = str(OUT_ROOT / f"z0_60um_w0eff_{W0_EFF*1e6:.1f}um")
# res_B = ff.load_scenario_npz(OUT_B)
# if res_B is None:
#     res_B = run(Nz=GRID["Nz"], Nt=GRID["Nt"], Nr=GRID["Nr"], R_factor=GRID["R_factor"],
#                 begin=BEGIN_M, end=END_M, save_stride=GRID["save_stride"],
#                 ckpt_every=200, verbose=True, wavelength=WAVELENGTH_M,
#                 energy_uJ=ENERGY_IN_GLASS_UJ, w0=W0_EFF, delta_t=DELTA_T_S,
#                 n2=N2, Ui_eV=UI_EV, meff_rel=MEFF_REL, tau_c=TAU_C_S, tau_r=TAU_R_S,
#                 rho_max=RHO_MAX_CM3, Us_eV=US_EV, tau_ste=TAU_STE_S,
#                 f_R=F_R, tau_d=TAU_D_S, tau_s=TAU_S_S, enable_ste=True,
#                 lambda_probe=LAMBDA_PROBE_M, rho_t_stride=GRID["rho_t_stride"],
#                 rho_r_stride=GRID["rho_r_stride"], out_dir=OUT_B,
#                 envelope="gaussian_focused")
# ff.plot_delay_series(res_B, DELAYS_FS, lambda_probe_m=LAMBDA_PROBE_M, E_tr_eV=E_TR_EV,
#                      n2=N2, tau_r_s=TAU_R_S, tau_ste_s=TAU_STE_S,
#                      z_face_um=None, x_half_um=15.0,
#                      save=str(FIG_DIR/"delay_series_w0eff.png"));